In [1]:
import json
import os
from openai import OpenAI
import openai
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import ttest_ind

In [4]:
def get_embedding(text, client, model="text-embedding-3-small"):
    response = client.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [2]:
def average_similarity(filename, api_key) :
    os.environ["OPENAI_API_KEY"] = api_key
    client = OpenAI()
    idea_files = [f'{filename}{i}.json' for i in ['', '2', '3']]
    idea_sets = []
    for idea_file in idea_files:
        with open(os.path.join('../templates/ppo_with_folds', idea_file), 'r') as f:
            ideas = json.load(f)
            idea_sets.append(ideas)
    seed_ideas = ['fold_specific_learning_rates', 'input_dependent_hyperplane_combination']
    embeddings = np.array([[get_embedding(idea["Experiment"], client) for idea in ideas if idea["Name"] not in seed_ideas] for ideas in idea_sets])
    similarities_between_sets = []
    for i in range(len(idea_sets)):
        for j in range(i+1, len(idea_sets)):
            similarities_between_sets.extend(cosine_similarity(embeddings[i], embeddings[j]).flatten())
    similarities_between_sets = np.array(similarities_between_sets)
    similarities_within_sets = []
    for i in range(len(idea_sets)):
        baseline_similarities = cosine_similarity(embeddings[i], embeddings[i])
        upper_triangle_indices = np.triu_indices(baseline_similarities.shape[0], k=1)
        upper_triangle_values = baseline_similarities[upper_triangle_indices]
        similarities_within_sets.extend(upper_triangle_values)
    similarities_within_sets = np.array(similarities_within_sets)
    np.save(f'{filename}_between_sets.npy', similarities_between_sets)
    np.save(f'{filename}_within_sets.npy', similarities_within_sets)
    print(f"Between set similarity: {similarities_between_sets.mean()}")
    print(f"Within set similarity: {similarities_within_sets.mean()}")
    t_stat, p_value = ttest_ind(similarities_within_sets, similarities_between_sets)
    print(f"T-statistic: {t_stat}, P-value: {p_value}")
    return similarities_between_sets, similarities_within_sets   

In [ ]:
between, within = average_similarity('high_temp_baseline', api_key)

Between set similarity: 0.6135337161528636
Within set similarity: 0.6361047071621124
T-statistic: 1.8456820395956648, P-value: 0.06601712701093315


In [6]:
low_temp_between = np.load('baseline_similarities_between_sets.npy')
low_temp_within = np.load('baseline_similarities_within_sets.npy')
print(f"p val for between set similarity: {ttest_ind(low_temp_between, between).pvalue}")
print(f"p val for within set similarity: {ttest_ind(low_temp_within, within).pvalue}")

p val for between set similarity: 0.011907515160885163
p val for within set similarity: 0.042042038456901194


In [18]:
# print the means
print("Baseline similarities between sets mean:", np.mean(baseline_similarities_between_sets))
print("Baseline similarities within sets mean:", np.mean(baseline_similarities_within_sets))
print("Persona similarities between sets mean:", np.mean(persona_similarities_between_sets))
print("Persona similarities within sets mean:", np.mean(persona_similarities_within_sets))
print("Elite similarities between sets mean:", np.mean(elite_similarities_between_sets))
print("Elite similarities within sets mean:", np.mean(elite_similarities_within_sets))
print("Elite System similarities between sets mean:", np.mean(system_similarities_between_sets))
print("Elite System similarities within sets mean:", np.mean(system_similarities_within_sets))
print("Simple system similarities between sets mean:", np.mean(simple_system_similarities_between_sets))
print("Simple system similarities within sets mean:", np.mean(simple_system_similarities_within_sets))

Baseline similarities between sets mean: 0.6358456466187324
Baseline similarities within sets mean: 0.6679140279606843
Persona similarities between sets mean: 0.6197223358900504
Persona similarities within sets mean: 0.6359698908043279
Elite similarities between sets mean: 0.6491274156882405
Elite similarities within sets mean: 0.6761115699602248
Elite System similarities between sets mean: 0.6220227168283712
Elite System similarities within sets mean: 0.6265273125424928
Simple system similarities between sets mean: 0.6515914539924254
Simple system similarities within sets mean: 0.6528275248778725


In [13]:
# save all the similarities to npy files
np.save('baseline_similarities_between_sets.npy', baseline_similarities_between_sets)
np.save('baseline_similarities_within_sets.npy', baseline_similarities_within_sets)
np.save('persona_similarities_between_sets.npy', persona_similarities_between_sets)
np.save('persona_similarities_within_sets.npy', persona_similarities_within_sets)
np.save('elite_similarities_between_sets.npy', elite_similarities_between_sets)
np.save('elite_similarities_within_sets.npy', elite_similarities_within_sets)
np.save('system_similarities_between_sets.npy', system_similarities_between_sets)
np.save('system_similarities_within_sets.npy', system_similarities_within_sets)
np.save('simple_system_similarities_between_sets.npy', simple_system_similarities_between_sets)
np.save('simple_system_similarities_within_sets.npy', simple_system_similarities_within_sets)

In [19]:
baseline = ttest_ind(baseline_similarities_within_sets, baseline_similarities_between_sets, equal_var=False).pvalue
persona = ttest_ind(persona_similarities_within_sets, persona_similarities_between_sets, equal_var=False).pvalue
elite = ttest_ind(elite_similarities_within_sets, elite_similarities_between_sets, equal_var=False).pvalue
system = ttest_ind(system_similarities_within_sets, system_similarities_between_sets, equal_var=False).pvalue
simple_system = ttest_ind(simple_system_similarities_within_sets, simple_system_similarities_between_sets, equal_var=False).pvalue
print("Baseline t-test p-value:", baseline)
print("Persona t-test p-value:", persona)
print("Elite t-test p-value:", elite)
print("System t-test p-value:", system)
print("Simple system t-test p-value:", simple_system)
between_sets = ttest_ind(baseline_similarities_between_sets, persona_similarities_between_sets, equal_var=True).pvalue
within_sets = ttest_ind(baseline_similarities_within_sets, persona_similarities_within_sets, equal_var=True).pvalue
elite_vs_baseline_between = ttest_ind(baseline_similarities_between_sets, elite_similarities_between_sets, equal_var=True).pvalue
elite_vs_baseline_within = ttest_ind(baseline_similarities_within_sets, elite_similarities_within_sets, equal_var=True).pvalue
system_vs_baseline_between = ttest_ind(baseline_similarities_between_sets, system_similarities_between_sets, equal_var=True).pvalue
system_vs_baseline_within = ttest_ind(baseline_similarities_within_sets, system_similarities_within_sets, equal_var=True).pvalue
simple_system_vs_baseline_between = ttest_ind(baseline_similarities_between_sets, simple_system_similarities_between_sets, equal_var=True).pvalue
simple_system_vs_baseline_within = ttest_ind(baseline_similarities_within_sets, simple_system_similarities_within_sets, equal_var=True).pvalue
elite_vs_system_between = ttest_ind(system_similarities_between_sets, elite_similarities_between_sets, equal_var=True).pvalue
elite_vs_system_within = ttest_ind(system_similarities_within_sets, elite_similarities_within_sets, equal_var=True).pvalue
persona_vs_simple_system_between = ttest_ind(simple_system_similarities_between_sets, persona_similarities_between_sets, equal_var=True).pvalue
persona_vs_simple_system_within = ttest_ind(simple_system_similarities_within_sets, persona_similarities_within_sets, equal_var=True).pvalue
print("persona vs baseline Between sets t-test p-value:", between_sets)
print("persona vs baseline Within sets t-test p-value:", within_sets)
print("elite vs baseline Between sets t-test p-value:", elite_vs_baseline_between)
print("elite vs baseline Within sets t-test p-value:", elite_vs_baseline_within)
print("system vs baseline Between sets t-test p-value:", system_vs_baseline_between)
print("system vs baseline Within sets t-test p-value:", system_vs_baseline_within)
print("simple system vs baseline Between sets t-test p-value:", simple_system_vs_baseline_between)
print("simple system vs baseline Within sets t-test p-value:", simple_system_vs_baseline_within)
print("elite vs system Between sets t-test p-value:", elite_vs_system_between)
print("elite vs system Within sets t-test p-value:", elite_vs_system_within)
print("persona vs simple system Between sets t-test p-value:", persona_vs_simple_system_between)
print("persona vs simple system Within sets t-test p-value:", persona_vs_simple_system_within)

Baseline t-test p-value: 0.014344995327748128
Persona t-test p-value: 0.2497915287935407
Elite t-test p-value: 0.04657248314620527
System t-test p-value: 0.6923648477502705
Simple system t-test p-value: 0.9291516923405476
persona vs baseline Between sets t-test p-value: 0.0949771817448227
persona vs baseline Within sets t-test p-value: 0.054394952518134206
elite vs baseline Between sets t-test p-value: 0.15197088179688004
elite vs baseline Within sets t-test p-value: 0.6132672003436839
system vs baseline Between sets t-test p-value: 0.10488184182479157
system vs baseline Within sets t-test p-value: 0.006290435578262499
simple system vs baseline Between sets t-test p-value: 0.07673405886722286
simple system vs baseline Within sets t-test p-value: 0.3691312082297019
elite vs system Between sets t-test p-value: 0.004917307355804473
elite vs system Within sets t-test p-value: 0.0009787208437534857
persona vs simple system Between sets t-test p-value: 0.002047732100998353
persona vs simple 